# Lab 8: Introduction au Framework ADK et Multi-Provider

**Navigation** : [Index](../../README.md) | [Précédent <<](../../Track1-LangChain/Day3-Data-Agents/Labs/Lab7-Data-Analysis-Agent/Lab7-Data-Analysis-Agent.ipynb) | [Suivant >>](Lab9-First-ADK-Agent.ipynb)

## Objectifs d'apprentissage

A la fin de ce laboratoire, vous saurez :
1. Expliquer l'architecture du Google Agent Development Kit (ADK)
2. Configurer un environnement multi-provider (Gemini, vLLM, OpenAI)
3. Créer un premier client LLM avec votre provider choisi
4. Comparer les réponses de différents providers sur le même prompt

### Prérequis
- Python 3.10+
- Fichier `.env` configuré avec au moins un provider (voir section Configuration)
- Connaissance de base des LLMs (API, tokens, temperature)

### Durée estimée : 45-60 minutes

***

## Configuration requise

Avant de commencer, assurez-vous d'avoir configuré votre fichier `.env` :

```bash
# Exemple de configuration .env
ACTIVE_PROVIDER=gemini  # ou vllm, openai

# Gemini (optionnel)
GOOGLE_API_KEY=your_gemini_key

# vLLM (optionnel)
VLLM_BASE_URL=https://your-vllm-endpoint.com/v1
VLLM_MODEL=Qwen/Qwen2.5-72B-Instruct

# OpenAI (optionnel)
OPENAI_API_KEY=sk-...
```

***

## 1. Architecture Google ADK

Le **Agent Development Kit (ADK)** de Google est un framework pour construire des agents IA avec :

> **Repères bibliographiques.** Le concept d'agent IA à base de LLM (LLM-as-agent : perception → raisonnement → action via *tools*) est formalisé dans la synthèse de référence Z. Xi et al., *The Rise and Potential of Large Language Model Based Agents: A Survey*, arXiv:2309.07864, 2023. Le framework **Google ADK** (Agent Development Kit) en fournit une implémentation officielle (`google.github.io/adk-docs`, dépôt `google/adk-python`), et **LangChain** — lancé en octobre 2022 par H. Chase — popularise l'approche modulaire (composabilité de chaînes, *tools*, mémoire) sur laquelle s'appuie une large part de l'écosystème agent Python.

- **Agents** : Entités qui interagissent via des tools
- **Tools** : Fonctions que l'agent peut appeler
- **Sessions** : Gestion de l'état conversationnel
- **Memory** : Persistance du contexte

### Comparaison avec LangChain

| Aspect | ADK | LangChain |
|--------|-----|----------|
| **Philosophie** | Google-first, intégré GCP | Multi-provider natif |
| **Agents** | Agent classes avec tools | Runnable, chains, agents |
| **Mémoire** | Session state intégrée | Modules séparés |
| **Déploiement** | Vertex AI Agent Engine | Variable |
| **Providers** | Gemini natif, OpenAI compatible | 50+ providers |

## 2. Configuration de l'Environnement

### Installation des dépendances

In [1]:
# Installation des dependances (decommenter si necessaire)
# !pip install google-adk google-genai litellm pydantic-settings

Import des modules et verification de l'installation.

In [2]:
import sys
import warnings
import os
from pathlib import Path

# Chemin vers Track2-GoogleADK depuis le répertoire de travail
track_dir = Path(os.getcwd()) / 'MyIA.AI.Notebooks' / 'ML' / 'DataScienceWithAgents' / 'Track2-GoogleADK'
sys.path.insert(0, str(track_dir))

warnings.filterwarnings('ignore', message=r'.*EXPERIMENTAL.*', category=UserWarning, module=r'google.adk')

from config import get_settings, get_provider_config
from utils.adk_runtime import build_agent, run_agent_turn

print('Modules importes avec succes')

Modules importes avec succes


### Vérification de la configuration

In [3]:
settings = get_settings()
config = get_provider_config(settings)

print(f'Provider actif: {config.provider.value}')
print(f'Modele: {config.model}')
print(f'Base URL: {config.base_url}')
api_display = '***' if config.api_key else 'Non requis (local)'
print(f'Cle API: {api_display}')

Provider actif: openrouter
Modele: openai/gpt-4.1-mini
Base URL: https://openrouter.ai/api/v1
Cle API: ***


Configuration detaillee du provider LLM.

In [4]:
print('Configuration complete :')
print(f'  Provider: {config.provider.value}')
print(f'  Model: {config.model}')
print(f'  API Key: {"Set" if config.api_key else "Not required"}')
print(f'  Base URL: {config.base_url}')

Configuration complete :
  Provider: openrouter
  Model: openai/gpt-4.1-mini
  API Key: Set
  Base URL: https://openrouter.ai/api/v1


## 3. Premier Test avec le Client LLM

Utilisons notre couche d'abstraction pour envoyer un prompt simple.

In [5]:
agent_intro = build_agent(
    name='lab8_intro',
    description='Agent d introduction ADK',
    instruction='Assistant expert ADK, reponds de maniere concise et precise.',
    config=config
)

response = await run_agent_turn(agent_intro, 'Explique en 2 phrases l architecture ADK.')

print('Reponse :')
print(response.response_text)

Reponse :
L’architecture ADK se compose d’un noyau central modulable permettant l’intégration et la communication entre différents modules fonctionnels. Elle facilite ainsi le développement rapide d’applications grâce à une organisation claire et une interopérabilité optimisée.


Test de generation simple avec le client configure.

In [6]:
response2 = await run_agent_turn(
    agent_intro,
    'Quels sont les 3 composants principaux d un agent ADK ?'
)

print('Composants ADK :')
print(response2.response_text)

Composants ADK :
Les 3 composants principaux d’un agent ADK sont :  
1. **Capteurs** – pour percevoir l’environnement.  
2. **Module de décision** – pour traiter les perceptions et prendre des décisions.  
3. **Effecteurs** – pour agir sur l’environnement.


### Test avec prompt système

In [7]:
agent_system = build_agent(
    name='lab8_system',
    description='Agent avec instruction systeme',
    instruction='Tu es un expert en IA. Explique les concepts de maniere claire.',
    config=config
)

response3 = await run_agent_turn(
    agent_system,
    'Explique le concept de Session dans ADK.'
)

print('Session dans ADK :')
print(response3.response_text)

Session dans ADK :
Bien sûr !

Dans le contexte d’**ADK** (Azure Dev Kit, ou autre framework appelé ADK selon le contexte), le concept de **Session** désigne généralement une instance temporaire et isolée d’interaction entre un utilisateur (ou un système) et une application ou un service.

### Concept général de **Session** dans ADK

1. **Définition :**  
   Une session correspond à un ensemble d’échanges ou d’opérations qui ont lieu entre le client (utilisateur, appareil, service) et l’application basée sur ADK, durant une période donnée. Elle commence lorsqu’une interaction démarre et se termine lorsqu’elle est explicitement fermée ou expire.

2. **Gestion d’état :**  
   La session permet de maintenir un état entre plusieurs requêtes. Par exemple, lorsque tu interagis avec un service ADK, la session conserve des informations comme :
   - Identifiants d’authentification,
   - Données temporaires de la conversation ou opération,
   - Paramètres utilisateur,
   - Contexte lié à l’opéra

## 4. Comparaison Multi-Provider

Testons le même prompt avec différents providers pour comparer les réponses.

In [8]:
from config.providers import ProviderType

print('Providers disponibles:')
for p in ProviderType:
    print(f'  - {p.value}')

r_mp = await run_agent_turn(agent_intro, 'Quelle est la difference entre un LLM et un agent ?')
print('Reponse:', r_mp.response_text[:200], '...')

Providers disponibles:
  - gemini
  - openai
  - openrouter
  - vllm
  - lmstudio


Reponse: Un LLM (Large Language Model) est un modèle de traitement du langage naturel capable de générer du texte, comprendre des questions, et produire des réponses basées sur des données textuelles.

Un agen ...


### Interprétation

La réponse ci-dessus provient du provider configuré dans `.env` (`ACTIVE_PROVIDER`).

Pour tester un autre provider, modifiez votre fichier `.env` et redémarrez le kernel, ou instanciez un client avec une configuration explicite :

```python
from config import ProviderConfig, ProviderType

# Exemple pour vLLM
vllm_config = ProviderConfig(
    provider=ProviderType.VLLM,
    model="Qwen/Qwen2.5-72B-Instruct",
    base_url="https://your-vllm-endpoint.com/v1",
    api_key=None
)
vllm_client = LLMClient(vllm_config)
```

## 5. Interface de Chat avec Historique

Le client supporte également une interface de chat avec historique des messages.

In [9]:
sid = 'lab8_chat_session'

r_a = await run_agent_turn(agent_intro, 'Bonjour, je suis nouveau avec ADK.', session_id=sid)
r_b = await run_agent_turn(agent_intro, 'Peux-tu me rappeler ce que j ai dit ?', session_id=sid)
print('Tour 1:', r_a.response_text)
print('Tour 2:', r_b.response_text)

Tour 1: Bonjour ! Bienvenue avec ADK. Je suis là pour vous aider à démarrer et répondre à vos questions. Que souhaitez-vous savoir ou faire avec ADK ?
Tour 2: Vous m'avez demandé de vous rappeler ce que vous avez dit. Cependant, vous n'avez pas encore communiqué d'informations spécifiques à rappeler.


## 6. Architecture des Frameworks DS-STAR et MLE-STAR

DS-STAR et MLE-STAR sont deux agents de référence (State-of-the-Art) conçus par Google Research pour la data science et l'ingénierie ML — ils incarnent le paradigme d'agent LLM *planner-coder* (décomposition de tâche → génération de code → exécution → raffinement) décrit par Xi et al. (2025) et appliqué à la compétition Kaggle et au cycle d'expérimentation ML.

Ce track Track2-GoogleADK intègre les frameworks de recherche Google :

### DS-STAR (Data Science Agent)

Architecture Planner-Coder-Verifier pour la data science autonome :

```mermaid
flowchart TD
    FA["File Analyzer"] --> P["Planner"]
    P --> C["Coder"]
    P --> V["Verifier"]
    C --> E["Executor"]
    E --> V
```

Le planificateur distribue le travail entre génération et vérification, tandis que l'exécuteur renvoie les résultats au vérificateur pour fermer la boucle de raffinement.

**Performance** : 45.2% accuracy sur DABStep benchmark

### MLE-STAR (ML Engineering Agent)

Extension avec recherche web et optimisation automatique :

- Web Search pour modèles SOTA
- Ablation studies ciblées
- Ensemble stratégies automatisées

**Performance** : 63.6% médailles sur MLE-Bench-Lite

## Exercice : Comparaison Multi-Provider

Maintenant que vous avez compris l'architecture, testez votre capacité à utiliser différents providers pour la même tâche.

In [10]:
test_prompts = [
    'Explique la temperature dans les LLMs.',
    'Quels sont les avantages de l approche agentique ?',
    'Donne un exemple d utilisation d ADK.'
]

for i, p in enumerate(test_prompts, 1):
    resp = await run_agent_turn(agent_intro, p)
    print(f'Prompt {i}: {resp.response_text[:100]}...')

Prompt 1: La température dans les LLMs (Large Language Models) est un paramètre qui contrôle la créativité et ...


Prompt 2: Les avantages de l’approche agentique sont :

1. **Autonomie** : les agents fonctionnent de manière ...


Prompt 3: Un exemple d’utilisation d’ADK (Agent de Développement de Konnaissance) est l’intégration dans un se...


## Exercice : Chat Multi-Tours avec Contexte Data Science

Utilisez l'interface `chat()` du client LLM pour construire une conversation specialisee en data science. L'objectif est de simuler un assistant qui guide un utilisateur dans son analyse de données étapes par étapes.

### Objectifs
1. Construire un historique de conversation avec 3 echanges
2. Utiliser un system prompt specialise data science
3. Observer comment le contexte précédent influence les reponses

**Indice :**
- Utilisez `client.chat(messages)` avec une liste de dictionnaires `{"rôle": "user"/"assistant", "content": "..."}`
- Commencez par un message system via le premier élément de la liste

In [11]:
ds_agent = build_agent(
    name='lab8_ds',
    description='Agent DS',
    instruction='Tu es un expert en Data Science.',
    config=config
)

ds_sid = 'lab8_ds_sess'

for q in ['Comment traiter les valeurs manquantes ?', 'Difference imputation moyenne et mediane ?', 'Recommandation ?']:
    resp = await run_agent_turn(ds_agent, q, session_id=ds_sid)
    print(q[:30], ':', resp.response_text[:100])

Comment traiter les valeurs ma : Le traitement des valeurs manquantes est une étape cruciale en data science, car les données incompl


Difference imputation moyenne  : La différence entre l'imputation par la moyenne et par la médiane réside dans la manière dont elles 


Recommandation ? : Pour vous fournir une recommandation pertinente, pourriez-vous préciser le contexte ou le domaine d'


## Exercice : Exploration des Paramètres de Generation

Experimentez avec les paramètres `temperature` et `max_tokens` pour comprendre leur impact sur la qualite des reponses d'un agent. L'objectif est de trouver les paramètres optimaux pour différentes tâches d'agent.

### Objectifs
1. Tester 3 valeurs de temperature (0.1, 0.7, 1.5) sur un prompt technique
2. Observer l'impact de `max_tokens` sur la longueur des reponses
3. Determiner les paramètres ideaux pour du code generation vs. du texte creatif

**Indice :**
- `client.generate(prompt, temperature=0.1, max_tokens=100)` pour contrôler la generation
- Temperature basse = reponses déterministes (ideal pour du code)
- Temperature haute = reponses creatives (ideal pour du brainstorming)

In [12]:
def dataset_stats(rows: int, cols: int, missing_pct: float) -> dict:
    total = rows * cols
    missing = int(total * missing_pct / 100)
    present = total - missing
    return {'total_cells': total, 'missing_cells': missing, 'present_cells': present, 'completeness': round(present/total*100, 2)}

sa = build_agent(
    name='lab8_stats',
    description='Agent stats',
    instruction='Utilise dataset_stats pour analyser.',
    tools=(dataset_stats,),
    config=config
)

sr = await run_agent_turn(sa, 'Analyse 1000 lignes x 10 colonnes avec 5 pct manquantes')
print('Resultat:', sr.response_text)
print('Tool invoked:', sr.tool_was_invoked)

Resultat: L'analyse d'un dataset de 1000 lignes et 10 colonnes avec 5 % de valeurs manquantes donne les résultats suivants :

- Nombre total de cellules : 10 000
- Nombre de cellules manquantes : 500
- Nombre de cellules présentes (non manquantes) : 9 500
- Taux de complétude (données présentes) : 95 %

Le dataset est donc majoritairement complet avec seulement 5 % de valeurs manquantes.
Tool invoked: True


## Résumé et Prochaines Étapes

### Ce que nous avons appris

1. **Configuration multi-provider** : Un seul fichier `.env` permet de switcher entre Gemini, vLLM, OpenAI
2. **Abstraction LiteLLM** : Interface unifiée pour tous les providers
3. **Client LLM simple** : `generate()` et `chat()` pour interagir avec n'importe quel modèle
4. **Architecture DS-STAR** : Framework Planner-Coder-Verifier pour la data science autonome

### Points clés à retenir

| Concept | Description |
|---------|-------------|
| `ProviderConfig` | Configuration d'un provider LLM |
| `LLMClient` | Client unifié pour tous les providers |
| `generate()` | Génération simple avec prompt |
| `chat()` | Conversation multi-tours avec historique |

### Prochaines étapes

- **Lab 9** : Créer un premier agent ADK avec tools Python pour analyser des DataFrames
- **Lab 10** : Implémenter le File Analyzer de DS-STAR
- **Lab 11** : Boucle Planner-Coder-Verifier

***

**Navigation** : [Index](../../README.md) | [Précédent <<](../../Track1-LangChain/Day3-Data-Agents/Labs/Lab7-Data-Analysis-Agent/Lab7-Data-Analysis-Agent.ipynb) | [Suivant >> Lab 9 - First ADK Agent](Lab9-First-ADK-Agent.ipynb)

## Ressources

- [Google ADK Documentation](https://github.com/google/adk-samples)
- [DS-STAR Paper](https://research.google/blog/ds-star-a-state-of-the-art-versatile-data-science-agent/)
- [MLE-STAR Paper](https://research.google/blog/mle-star-a-state-of-the-art-machine-learning-engineering-agents/)
- [LiteLLM Documentation](https://docs.litellm.ai/)

## Références

1. Z. Xi et al., *The Rise and Potential of Large Language Model Based Agents: A Survey*, arXiv:2309.07864, 2023. Synthèse de référence sur les agents IA à base de LLM (architecture perception-raisonnement-action, *tools*, mémoire, cadres planner-coder).
2. Google, *Agent Development Kit (ADK)*, documentation officielle, `google.github.io/adk-docs` (dépôt `google/adk-python`). Framework agent-first de Google (sessions, mémoire, Vertex AI Agent Engine).
3. H. Chase, *LangChain*, octobre 2022, `langchain.com` / `github.com/langchain-ai/langchain`. Framework modulaire open-source pour applications LLM (chaînes composables, *tools*, mémoire) — référence de l'écosystème agent Python.
4. Google Research, *DS-STAR: A State-of-the-Art Versatile Data Science Agent*, 2025, `research.google/blog/ds-star-a-state-of-the-art-versatile-data-science-agent/`. Agent SOTA data science (paradigme planner-coder).
5. Google Research, *MLE-STAR: A State-of-the-Art Machine Learning Engineering Agent*, 2025, `research.google/blog/mle-star-a-state-of-the-art-machine-learning-engineering-agents/`. Agent SOTA ingénierie ML (cycle d'expérimentation, Kaggle).